# 2. Depuración, ingeniería de características y partición

In [1]:
from config import *

crudo = pd.read_csv(
    DATOS_CRUDOS / "diabetic_data.csv",
    na_values=["?"],
    keep_default_na=False,
    low_memory=False,
)
print(f"Punto de partida: {crudo.shape[0]:,} filas x {crudo.shape[1]} columnas")

Punto de partida: 101,766 filas x 50 columnas


## 2.1. Construcción de la variable objetivo

### Distribución original en tres niveles

In [2]:
distribucion = crudo["readmitted"].value_counts().rename("encuentros").to_frame()
distribucion["%"] = (100 * crudo["readmitted"].value_counts(normalize=True)).round(2)
distribucion

,encuentros,%
readmitted,,
NO,54864,53.91
>30,35545,34.93
<30,11357,11.16


### Exclusión de altas incompatibles con una readmisión

Un paciente que fallece durante la hospitalización o que es trasladado a cuidados paliativos no puede reingresar. Conservar esos registros introduce ruido irreducible: quedan etiquetados como "no readmitido" por una razón ajena al mecanismo clínico que se quiere modelar.

In [5]:
DESTINOS_EXCLUIDOS = [11, 13, 14, 19, 20, 21]
excluidos = crudo["discharge_disposition_id"].isin(DESTINOS_EXCLUIDOS)

comparacion = pd.DataFrame({
    "encuentros": [excluidos.sum(), (~excluidos).sum()],
    "% del total": [
        100 * excluidos.mean(),
        100 * (~excluidos).mean(),
    ],
    "tasa readmisión < 30 días (%)": [
        100 * (crudo.loc[excluidos, "readmitted"] == "<30").mean(),
        100 * (crudo.loc[~excluidos, "readmitted"] == "<30").mean(),
    ],
}, index=["Fallecimiento / hospicio", "Resto"]).round(2)
comparacion

,encuentros,% del total,tasa readmisión < 30 días (%)
Fallecimiento / hospicio,2423,2.38,1.77
Resto,99343,97.62,11.39


In [4]:
ETIQUETAS_EXCLUIDOS = {
    11: "Fallecido",
    13: "Hospicio / domicilio",
    14: "Hospicio / institución",
    19: "Fallecido en domicilio (hospicio)",
    20: "Fallecido en institución (hospicio)",
    21: "Fallecido, lugar desconocido (hospicio)",
}

desglose = (crudo.loc[excluidos]
            .groupby("discharge_disposition_id")["readmitted"]
            .value_counts().unstack(fill_value=0))
desglose["total"] = desglose.sum(axis=1)
desglose["% < 30 días"] = (100 * desglose["<30"] / desglose["total"]).round(2)
desglose.index = [ETIQUETAS_EXCLUIDOS[i] for i in desglose.index]
desglose

readmitted,<30,>30,NO,total,% < 30 días
Fallecido,0,0,1642,1642,0.00
Hospicio / domicilio,19,36,344,399,4.76
Hospicio / institución,24,7,341,372,6.45
Fallecido en domicilio (hospicio),0,0,8,8,0.00
Fallecido en institución (hospicio),0,0,2,2,0.00


Los encuentros con alta por fallecimiento u hospicio son el 2.38 % del total y presentan una tasa de readmisión temprana del 1.77 %, frente al 11.16 % del resto. El desglose por código separa dos situaciones distintas. Entre los 1,652 encuentros con alta por fallecimiento no hay ni un solo caso de readmisión, como exige la lógica: el evento es imposible. Los 43 casos que componen ese 1.77 % residual provienen en su totalidad de los 771 encuentros con alta a cuidados paliativos, donde un reingreso sí es posible por una crisis de dolor o una complicación aguda. La razón para excluirlos no es entonces la imposibilidad del evento, sino la población objetivo: son pacientes en un régimen de atención orientado al confort y no a evitar el reingreso, de modo que la intervención que este modelo pretende orientar no les es aplicable.

## 2.2. Aplicación de los filtros con trazabilidad

In [15]:
traza = []


def registrar(etapa, datos):
    """Añade una fila al registro de trazabilidad del proceso de depuración."""
    traza.append({
        "etapa": etapa,
        "filas": len(datos),
        "columnas": datos.shape[1],
        "pacientes": datos["patient_nbr"].nunique(),
    })


limpio = crudo.copy()
registrar("0. Dataset crudo", limpio)

# --- Filtro 1: altas incompatibles con una readmisión --------------------
limpio = limpio[~limpio["discharge_disposition_id"].isin(DESTINOS_EXCLUIDOS)]
registrar("1. Sin fallecimiento ni hospicio", limpio)

# --- Filtro 2: género inválido ------------------------------------------
limpio = limpio[limpio["gender"] != "Unknown/Invalid"]
registrar("2. Sin género inválido", limpio)

pd.DataFrame(traza)

,etapa,filas,columnas,pacientes
0,0. Dataset crudo,101766,50,71518
1,1. Sin fallecimiento ni hospicio,99343,50,69990
2,2. Sin género inválido,99340,50,69987


In [16]:
IDENTIFICADOR = "patient_nbr"

# Las cohortes se construyen desde el crudo, aplicando solo las exclusiones
# clínicas, para poder comparar antes de decidir la unidad de análisis.
base = crudo[~crudo["discharge_disposition_id"].isin(DESTINOS_EXCLUIDOS)]
base = base[base["gender"] != "Unknown/Invalid"]
ordenado = base.sort_values("encounter_id")

cohortes = {
    "Todos los encuentros": base,
    "Uno aleatorio por paciente": (base.sample(frac=1, random_state=RANDOM_STATE)
                                       .drop_duplicates(IDENTIFICADOR)),
    "Primer encuentro por paciente": ordenado.drop_duplicates(IDENTIFICADOR,
                                                              keep="first"),
    "Último encuentro por paciente": ordenado.drop_duplicates(IDENTIFICADOR,
                                                              keep="last"),
}

comparacion_cohortes = pd.DataFrame({
    "registros": {k: len(v) for k, v in cohortes.items()},
    "pacientes": {k: v[IDENTIFICADOR].nunique() for k, v in cohortes.items()},
    "tasa positiva (%)": {k: 100 * (v["readmitted"] == "<30").mean()
                          for k, v in cohortes.items()},
}).round(2)
comparacion_cohortes

,registros,pacientes,tasa positiva (%)
Todos los encuentros,99340,69987,11.39
Uno aleatorio por paciente,69987,69987,7.44
Primer encuentro por paciente,69987,69987,8.98
Último encuentro por paciente,69987,69987,4.96


In [18]:
frecuencia = base[IDENTIFICADOR].value_counts()
primero = ordenado.drop_duplicates(IDENTIFICADOR, keep="first")
ultimo = ordenado.drop_duplicates(IDENTIFICADOR, keep="last")

varios_primero = primero[IDENTIFICADOR].map(frecuencia) > 1
varios_ultimo = ultimo[IDENTIFICADOR].map(frecuencia) > 1

sesgo_seleccion = pd.DataFrame({
    "pacientes": [
        (~varios_primero).sum(),
        varios_primero.sum(),
        varios_ultimo.sum(),
    ],
    "tasa positiva (%)": [
        100 * (primero.loc[~varios_primero, "readmitted"] == "<30").mean(),
        100 * (primero.loc[varios_primero, "readmitted"] == "<30").mean(),
        100 * (ultimo.loc[varios_ultimo, "readmitted"] == "<30").mean(),
    ],
}, index=["Un solo encuentro",
          "Primer encuentro de pacientes con varios",
          "Último encuentro de pacientes con varios"]).round(2)
sesgo_seleccion["% de la cohorte"] = (
    100 * sesgo_seleccion["pacientes"] / base[IDENTIFICADOR].nunique()
).round(1)
sesgo_seleccion

,pacientes,tasa positiva (%),% de la cohorte
Un solo encuentro,53646,4.26,76.7
Primer encuentro de pacientes con varios,16341,24.47,23.3
Último encuentro de pacientes con varios,16341,7.23,23.3


### Elección de la unidad de análisis

El capítulo 1 mostró que los encuentros de un mismo paciente no son independientes y que la posición del encuentro en el historial predice el desenlace. Hay cuatro formas de responder a ese problema y la elección afecta los resultados de forma sustancial, así que se documenta con evidencia.

| Estrategia | Registros | Tasa positiva | Problema |
|---|---|---|---|
| **Todos los encuentros (adoptada)** | 99,340 | 11.39 % | Requiere agrupar la validación por paciente |
| Un encuentro aleatorio por paciente | 69,987 | 7.44 % | Descarta el 30 % de los episodios |
| Primer encuentro por paciente | 69,987 | 8.98 % | Condiciona sobre la existencia de un episodio posterior |
| Último encuentro por paciente | 69,987 | 4.96 % | Censura a la derecha |

Que la misma cohorte produzca tasas entre el 4.96 % y el 11.39 % según el episodio que se elija no es variabilidad de muestreo: es sesgo de selección. El desglose lo hace explícito.

| Grupo | Tasa positiva |
|---|---|
| Pacientes con un solo encuentro (76.7 %) | 4.26 % |
| Primer encuentro de pacientes con varios | 24.47 % |
| Último encuentro de pacientes con varios | 7.23 % |

Seleccionar el primer encuentro de un paciente con varios **condiciona sobre el futuro**: por construcción ese paciente volvió al hospital, y lo único que decide la etiqueta es si volvió antes o después de los 30 días. De ahí el 24.47 %, casi seis veces la tasa de los pacientes con un solo episodio. La cohorte resultante mezcla entonces dos subpoblaciones cuya diferencia es un artefacto del muestreo y no riesgo clínico. Simétricamente, quedarse con el último encuentro censura a la derecha: los reingresos posteriores al cierre de la ventana de observación en 2008 no se registran y esos casos se etiquetan como no readmitidos.

Se adoptan por tanto **todos los encuentros**, con `StratifiedGroupKFold` agrupado por `patient_nbr` en cada partición y en cada fold de la validación cruzada. Las razones:

1. **No introduce sesgo de selección.** Ningún episodio se descarta y ninguna etiqueta depende de información posterior al alta que se está evaluando.
2. **La prevalencia es la correcta.** El 11.39 % es la tasa de readmisión a 30 días por episodio, que es exactamente la cantidad que los sistemas de salud miden y penalizan. Una tasa por paciente respondería a una pregunta distinta.
3. **La unidad de análisis coincide con la unidad de decisión.** El modelo se aplicaría en el momento de un alta concreta, no a un paciente en abstracto.
4. **El agrupamiento resuelve la dependencia sin coste añadido.** `StratifiedGroupKFold` mantiene a cada paciente íntegramente dentro de un mismo fold, de modo que el modelo nunca se evalúa sobre un paciente que ya vio en entrenamiento. Su costo computacional es idéntico al de `StratifiedKFold`; el incremento es únicamente el 42 % de filas adicionales.

Se descarta la alternativa de colapsar el historial en una fila por paciente con variables agregadas. Tiene tres impedimentos en este dataset: la variable objetivo perdería su definición por episodio, cualquier agregado calculado sobre todos los episodios incorporaría información posterior al evento a predecir, y el 76.7 % de los pacientes tiene un único encuentro, por lo que esas variables serían constantes para tres de cada cuatro filas. Además, el dataset no incluye fechas, así que los intervalos entre ingresos no son calculables. Las variables `number_inpatient`,

### Por qué la posición del encuentro no se incorpora como predictor

El capítulo 1 mostró que la posición del encuentro en el historial del paciente se asocia con fuerza al desenlace: la tasa de readmisión pasa del 9.0 % en el primer episodio registrado al 23.9 % a partir del cuarto. Esa evidencia se usó para justificar el agrupamiento de la validación por paciente, pero la variable **no se incorpora al conjunto de predictores**, por tres razones.

La primera es de definición. La posición no mide el historial clínico del paciente, sino su historial *dentro de esta muestra*: 130 instituciones, diez años de registro y solo los episodios que cumplieron los criterios de inclusión del dataset. Un paciente con veinte hospitalizaciones previas en otra red recibiría el valor cero.

La segunda es el truncamiento por la izquierda. Un paciente cuyo primer ingreso ocurrió antes de 1999 aparece con posición cero en su primer episodio registrado, aunque no fuera realmente el primero. El valor cero resulta entonces ambiguo: agrupa a los pacientes sin historial previo con aquellos cuyo historial cae fuera de la ventana de observación, y el modelo no puede distinguirlos.

La tercera es la redundancia con una variable mejor definida. `number_inpatient` mide las hospitalizaciones del paciente en el año previo a cada episodio, con una ventana fija e idéntica para todas las filas e independiente de dónde comience el registro. Captura el mismo concepto clínico —historial de hospitalización— sin la ambigüedad anterior, y el capítulo 4 confirma que es el predictor numérico con mayor tamaño de efecto del conjunto.

## 2.4. Variable objetivo y eliminación de columnas

### 2.4.1. Columnas descartadas

Las eliminaciones responden a motivos distintos y cada una se justifica por separado.

**`readmitted`.** Es la variable original de tres niveles, de la que se deriva el objetivo binario en la celda anterior. Conservarla dejaría el desenlace entre los predictores, la forma más directa de fuga de información posible: cualquier modelo alcanzaría un AUC perfecto sin haber aprendido nada. Se elimina después de derivar `readmitted_30`, no antes.

**`encounter_id`.** Identificador secuencial del episodio, sin contenido clínico. Su eliminación no solo evita ruido: al ser creciente en el tiempo, un modelo podría usarlo para captar tendencias del propio registro (cambios de codificación, incorporación de hospitales a la red) en lugar del mecanismo clínico que se quiere modelar.

**`weight`.** El 96.86 % de los valores está ausente. Imputar supondría generar un valor sintético para casi toda la muestra, de modo que la variable aportaría el artefacto de la imputación y no información del paciente. La sección 4.3 del capítulo 1 añade un argumento definitivo: su ausencia es la única del conjunto completamente independiente del desenlace (chi² nulo, p = 1.00), así que tampoco el hecho de faltar es informativo. Se descarta sin sustituto.

**`payer_code`.** El 39.56 % de valores ausentes y, sobre todo, su naturaleza: identifica al asegurador, no describe el estado clínico del paciente. Su asociación con la readmisión reflejaría diferencias de cobertura y acceso al sistema antes que riesgo médico, lo que introduciría un factor de confusión difícil de interpretar en un modelo destinado a orientar decisiones clínicas al alta. Se deja constancia en las limitaciones de que el contexto asegurador queda, por tanto, fuera del análisis.

**Los nueve fármacos con varianza nula o casi nula.** `acetohexamide`, `troglitazone`, `examide`, `citoglipton`, `glimepiride-pioglitazone`, `metformin-rosiglitazone` y `metformin-pioglitazone` son estrictamente constantes: todos los encuentros registran el valor `No`, por lo que su varianza es cero y no pueden discriminar entre clases. En `tolbutamide` y `glipizide-metformin` la variación se limita a una decena de casos sobre 99,340, cantidad insuficiente para estimar ningún efecto con estabilidad. Más allá de su inutilidad predictiva, conservarlos tiene un costo concreto: al aplicar `OneHotEncoder` dentro de cada fold generarían columnas casi vacías, y una categoría presente solo en el conjunto de entrenamiento de un fold produce una columna nula en el de validación, lo que desestabiliza los coeficientes de los modelos lineales entre repeticiones.

In [19]:
# --- Variable objetivo ---------------------------------------------------
limpio["readmitted_30"] = (limpio["readmitted"] == "<30").astype(int)

# --- Columnas descartadas (justificación en el capítulo 1, sección 5) ----
FARMACOS_CONSTANTES = [
    "acetohexamide", "troglitazone", "examide", "citoglipton", "tolbutamide",
    "glipizide-metformin", "glimepiride-pioglitazone",
    "metformin-rosiglitazone", "metformin-pioglitazone",
]
COLUMNAS_DESCARTADAS = ["encounter_id", "weight", "payer_code", "readmitted"]

limpio = limpio.drop(columns=COLUMNAS_DESCARTADAS + FARMACOS_CONSTANTES)
registrar("4. Columnas descartadas y target derivado", limpio)

pd.DataFrame(traza)

,etapa,filas,columnas,pacientes
0,0. Dataset crudo,101766,50,71518
1,1. Sin fallecimiento ni hospicio,99343,50,69990
2,2. Sin género inválido,99340,50,69987
3,4. Columnas descartadas y target derivado,99340,38,69987


La depuración lleva el conjunto de 101,766 encuentros a 69,987 pacientes únicos, y de 50 a 38 columnas. La pérdida de filas se explica casi por completo por el filtro de independencia (29,353 encuentros repetidos); las exclusiones clínicas suman 2,426 registros.

## 2.5. Ingeniería de características

Todas las variables derivadas se construyen con información disponible **en el momento del alta**, condición necesaria para que el modelo sea aplicable en la práctica clínica.

### 2.5.1. Agrupación de los diagnósticos ICD-9

Los tres códigos se agrupan por capítulo de la clasificación ICD-9, aislando la diabetes (250.xx) como grupo propio por ser el eje del estudio. Los códigos con prefijo `V` (factores que influyen en el estado de salud) y `E` (causas externas de lesión) van a la categoría residual. Los faltantes conservan una categoría propia, coherente con el tratamiento del resto de ausencias.

In [20]:
def agrupar_icd9(codigo):
    """Asigna un código ICD-9 a su capítulo clínico.

    El agrupamiento sigue el usado en la literatura sobre este dataset: se
    aísla la diabetes (250.xx) y el resto se asigna al capítulo ICD-9
    correspondiente, con una categoría residual para los capítulos poco
    frecuentes y los códigos suplementarios V y E.

    Parameters
    ----------
    codigo : str or float
        Código ICD-9 tal como aparece en el dataset.

    Returns
    -------
    str
        Nombre del capítulo clínico.
    """
    if pd.isna(codigo) or codigo == "":
        return "No registrado"

    codigo = str(codigo)
    if codigo.startswith(("V", "E")):
        return "Otro"
    if codigo.startswith("250"):
        return "Diabetes"

    try:
        valor = float(codigo)
    except ValueError:
        return "Otro"

    if 390 <= valor <= 459 or valor == 785:
        return "Circulatorio"
    if 460 <= valor <= 519 or valor == 786:
        return "Respiratorio"
    if 520 <= valor <= 579 or valor == 787:
        return "Digestivo"
    if 800 <= valor <= 999:
        return "Lesiones"
    if 710 <= valor <= 739:
        return "Musculoesquelético"
    if 580 <= valor <= 629 or valor == 788:
        return "Genitourinario"
    if 140 <= valor <= 239:
        return "Neoplasias"
    return "Otro"


for columna in ["diag_1", "diag_2", "diag_3"]:
    limpio[f"{columna}_grupo"] = limpio[columna].map(agrupar_icd9)

limpio = limpio.drop(columns=["diag_1", "diag_2", "diag_3"])

grupos_diag = pd.DataFrame({
    "principal": limpio["diag_1_grupo"].value_counts(normalize=True),
    "secundario": limpio["diag_2_grupo"].value_counts(normalize=True),
    "terciario": limpio["diag_3_grupo"].value_counts(normalize=True),
})
(100 * grupos_diag).round(2)

,principal,secundario,terciario
Circulatorio,29.88,31.36,29.80
Diabetes,8.72,12.79,17.09
Digestivo,9.40,4.12,3.88
Genitourinario,5.04,8.20,6.48
Lesiones,6.90,2.40,1.91
Musculoesquelético,4.97,1.77,1.91
Neoplasias,3.15,2.34,1.67
No registrado,0.02,0.36,1.43
Otro,17.91,26.20,28.78
Respiratorio,14.03,10.46,7.05


El agrupamiento reduce más de 700 códigos a diez categorías interpretables. El diagnóstico principal más frecuente es el circulatorio (30.6 %), seguido del respiratorio (13.6 %). La diabetes aparece como diagnóstico principal en solo el 8.2 % de los casos, lo que confirma que en esta cohorte suele ser una comorbilidad y no el motivo del ingreso.

### 2.5.2. Edad como variable ordinal

La edad viene agrupada en diez intervalos de amplitud constante. Al tener orden natural se codifica como entero, lo que evita generar diez columnas binarias y permite a los modelos lineales aprovechar la monotonía. La versión textual se descarta para no mantener dos representaciones de la misma información, que serían redundantes entre sí.

In [21]:
ORDEN_EDAD = [
    "[0-10)", "[10-20)", "[20-30)", "[30-40)", "[40-50)",
    "[50-60)", "[60-70)", "[70-80)", "[80-90)", "[90-100)",
]
ETIQUETAS_EDAD = {i: v for i, v in enumerate(ORDEN_EDAD)}

limpio["age_ord"] = limpio["age"].map({v: i for i, v in enumerate(ORDEN_EDAD)})
assert limpio["age_ord"].notna().all(), "Hay categorías de edad sin mapear"

limpio = limpio.drop(columns=["age"])
limpio["age_ord"].value_counts().sort_index().rename(ETIQUETAS_EDAD).to_frame("pacientes")

,pacientes
age_ord,
[0-10),160
[10-20),690
[20-30),1649
[30-40),3764
[40-50),9607
[50-60),17060
[60-70),22058
[70-80),25329
[80-90),16434


### 2.5.3. Historial de utilización de servicios

Se añade un indicador binario de hospitalización previa, que en la literatura clínica es uno de los predictores más robustos de reingreso.

**Nota metodológica.** Una primera versión de este trabajo incluía también `visitas_previas`, la suma de las tres variables de utilización. Se descartó como predictor: al ser una combinación lineal exacta de `number_outpatient`, `number_emergency` y `number_inpatient`, introduce colinealidad perfecta. El factor de inflación de la varianza de ese bloque se vuelve infinito y los coeficientes de cualquier modelo lineal quedan indeterminados. El indicador binario no tiene ese problema porque es una función no lineal de una sola variable, y aporta información distinta: separa el hecho de haber estado hospitalizado del número de veces.

In [22]:
limpio["hospitalizacion_previa"] = (limpio["number_inpatient"] > 0).astype(int)

utilizacion = pd.DataFrame({
    "% con valor cero": [
        100 * (limpio[c] == 0).mean()
        for c in ["number_outpatient", "number_emergency", "number_inpatient"]
    ],
    "media": [
        limpio[c].mean()
        for c in ["number_outpatient", "number_emergency", "number_inpatient"]
    ],
    "máximo": [
        limpio[c].max()
        for c in ["number_outpatient", "number_emergency", "number_inpatient"]
    ],
}, index=["number_outpatient", "number_emergency", "number_inpatient"]).round(2)
print(f"Pacientes con hospitalización previa: "
      f"{100 * limpio['hospitalizacion_previa'].mean():.2f} %")
utilizacion

Pacientes con hospitalización previa: 33.32 %


,% con valor cero,media,máximo
number_outpatient,83.54,0.37,42
number_emergency,88.83,0.20,76
number_inpatient,66.68,0.63,21


Las tres variables tienen exceso de ceros: entre el 87 % y el 93 % de los pacientes no registran visitas previas del tipo correspondiente. Esa característica justifica el indicador binario, que separa la señal principal del componente de conteo, más ruidoso.

### 2.5.4. Intensidad del tratamiento farmacológico

Se cuenta cuántos de los fármacos retenidos estaban activos durante el ingreso, entendiendo por activo cualquier estado distinto de `No`.

In [23]:
FARMACOS = [
    "metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride",
    "glipizide", "glyburide", "pioglitazone", "rosiglitazone", "acarbose",
    "miglitol", "tolazamide", "insulin", "glyburide-metformin",
]
assert all(f in limpio.columns for f in FARMACOS), "Fármaco no encontrado"

limpio["n_farmacos_activos"] = (
    limpio[FARMACOS].isin(["Up", "Down", "Steady"]).sum(axis=1)
)

print(f"Fármacos retenidos: {len(FARMACOS)}")
limpio["n_farmacos_activos"].value_counts().sort_index().to_frame("pacientes")

Fármacos retenidos: 14


,pacientes
n_farmacos_activos,
0,22636
1,46039
2,21575
3,7706
4,1321
5,58
6,5


### 2.5.5. Etiquetado explícito de las pruebas no realizadas

En `A1Cresult` y `max_glu_serum` el valor `None` significa que la prueba no se
realizó. Mantener esa cadena es frágil: cualquier lectura posterior del archivo
con la configuración por defecto de `pandas` la convertiría en valor nulo,
borrando en silencio la categoría más frecuente de ambas variables. Se renombra
a `"No medida"`, que es además más legible en las tablas y los gráficos.

In [24]:
PRUEBAS_LABORATORIO = ["A1Cresult", "max_glu_serum"]
for columna in PRUEBAS_LABORATORIO:
    limpio[columna] = limpio[columna].replace({"None": "No medida"})

pd.DataFrame({
    columna: limpio[columna].value_counts(normalize=True).round(4)
    for columna in PRUEBAS_LABORATORIO
})

,A1Cresult,max_glu_serum
>200,NaN,0.0143
>300,NaN,0.0120
>7,0.0380,NaN
>8,0.0819,NaN
No medida,0.8305,0.9481
Norm,0.0495,0.0256


## 2.6. Agrupación de categorías poco frecuentes

Las categorías con muy pocos casos causan dos problemas: producen columnas binarias casi vacías al codificar, con coeficientes inestables entre folds, y violan el supuesto de frecuencias esperadas de la prueba chi-cuadrado que se usará en el capítulo 4.

Se aplica un criterio único, uniforme y fijado a priori: **toda categoría con menos del 1 % de los casos se agrupa en `"Otro"`**. Se aplica a todas las variables categóricas, no solo a las de cardinalidad alta, porque el panel de fármacos y los grupos diagnósticos también contienen niveles con muy pocos casos. Los tres identificadores administrativos se traducen antes a su descripción, tanto para que los gráficos sean legibles como para que las etiquetas tengan sentido clínico.

Aplicar la regla de forma selectiva sería una decisión discrecional difícil de justificar ante un revisor; aplicarla igual a todas las variables hace el criterio auditable y verificable, como se comprueba en el capítulo 4 al validar los supuestos de la prueba chi-cuadrado.

Hay una excepción deliberada: **`race` no se agrupa**. Con la regla mecánica, la categoría `Asian` (0.6 % de los pacientes) se fundiría en una categoría residual y el grupo desaparecería del análisis. Dos razones lo desaconsejan. La estadística es que no hace falta: con cerca de 430 pacientes, la frecuencia esperada en la tabla de contingencia ronda las 38 observaciones, muy por encima del mínimo exigido. La sustantiva es que los atributos protegidos son precisamente los que se necesitan desagregados para auditar la equidad del modelo, de modo que agrupar el grupo minoritario iría en contra del objetivo. La excepción se declara en el código con una constante explícita, no se aplica en silencio.

In [25]:
def agrupar_poco_frecuentes(serie, umbral=0.01, etiqueta="Otro"):
    """Agrupa en una sola categoría los valores con frecuencia relativa baja.

    Parameters
    ----------
    serie : pandas.Series
        Variable categórica.
    umbral : float, default 0.01
        Frecuencia relativa mínima para conservar una categoría.
    etiqueta : str, default "Otro"
        Nombre de la categoría agrupada.

    Returns
    -------
    pandas.Series
        Serie con las categorías poco frecuentes reemplazadas.
    """
    frecuencias = serie.value_counts(normalize=True)
    conservadas = frecuencias[frecuencias >= umbral].index
    return serie.where(serie.isin(conservadas), etiqueta)


UMBRAL_CATEGORIA = 0.01
IDS_ADMINISTRATIVOS = [
    "admission_type_id", "discharge_disposition_id", "admission_source_id",
]

mapeos = {}
for linea in (DATOS_CRUDOS / "IDS_mapping.csv").read_text(
        encoding="utf-8").splitlines():
    linea = linea.strip()
    if not linea or linea == ",":
        continue
    if linea.endswith("_id,description"):
        clave = linea.split(",")[0]
        mapeos[clave] = {}
        continue
    codigo, _, descripcion = linea.partition(",")
    if codigo.isdigit():
        mapeos[clave][int(codigo)] = descripcion.strip('"')

# Los identificadores administrativos se traducen a su descripción antes de
# agrupar, para que las etiquetas resultantes sean legibles e interpretables.
for columna in IDS_ADMINISTRATIVOS:
    limpio[columna] = limpio[columna].map(mapeos[columna])

# medical_specialty: la ausencia se conserva como categoría explícita.
limpio["especialidad"] = limpio["medical_specialty"].fillna("No registrada")
limpio = limpio.drop(columns=["medical_specialty"])

# El umbral se aplica de forma uniforme a todas las variables categóricas,
# incluidos el panel de fármacos y los grupos diagnósticos, para no dejar
# ninguna categoría residual con frecuencias insuficientes.
# race queda exenta: sus categorías cumplen los supuestos estadísticos sin
# agrupar (se verifica en el capítulo 4) y agrupar un grupo demográfico
# minoritario en una categoría residual lo haría desaparecer del análisis de
# equidad del modelo, que es justo donde su información resulta necesaria.
EXENTAS_DE_AGRUPAR = ["race"]
candidatas = [c for c in columnas_texto(limpio) if c not in EXENTAS_DE_AGRUPAR]

resumen_agrupacion = []
for columna in candidatas:
    antes = limpio[columna].nunique()
    limpio[columna] = agrupar_poco_frecuentes(limpio[columna], UMBRAL_CATEGORIA)
    despues = limpio[columna].nunique()
    if antes != despues:
        resumen_agrupacion.append({
            "variable": columna,
            "categorías antes": antes,
            "categorías después": despues,
            "categoría menos frecuente después (%)": round(
                100 * limpio[columna].value_counts(normalize=True).iloc[-1], 2),
        })

print(f"Variables con categorías agrupadas: {len(resumen_agrupacion)} "
      f"de {len(candidatas)}")
pd.DataFrame(resumen_agrupacion)

Variables con categorías agrupadas: 18 de 26


,variable,categorías antes,categorías después,categoría menos frecuente después (%)
0,admission_type_id,8,6,0.35
1,discharge_disposition_id,21,8,1.19
2,admission_source_id,17,7,1.09
3,repaglinide,4,3,0.15
4,nateglinide,4,2,0.69
5,chlorpropamide,4,2,0.09
6,glimepiride,4,3,0.52
7,glipizide,4,3,1.32
8,glyburide,4,3,1.37
9,pioglitazone,4,3,0.35


El efecto es sustancial en las variables de cardinalidad alta: el destino al alta pasa de 21 a 9 categorías, la especialidad médica de 72 a 11, el tipo de admisión de 8 a 6 y la fuente de admisión de 17 a 7. En el panel de fármacos el efecto es menor y consiste en agrupar los niveles de ajuste de dosis con muy pocos casos. La insulina conserva sus cuatro niveles, porque todos superan el umbral. El capítulo 4 verifica que, tras esta agrupación, las frecuencias esperadas cumplen el supuesto de la prueba chi-cuadrado, lo que no ocurría antes: más de una cuarta parte de las celdas tenía frecuencia esperada inferior a 5.

### 2.6.1. Tratamiento de `race`

La variable `race` tiene un 2.23 % de faltantes. La decisión es **conservar la ausencia como categoría explícita**, no imputarla, por dos razones.

La estadística es que el capítulo 1 rechazó la ausencia completamente aleatoria: imputar la moda asignaría a todos los casos sin dato la categoría mayoritaria, creando una falsa homogeneidad.

La ética, más importante, es que `race` es un atributo protegido. Imputar la moda equivale a **atribuir a 1,917 pacientes una pertenencia racial que no declararon**, y esa atribución sintética se propagaría a cualquier análisis de equidad del modelo. Registrar "No informada" es la representación honesta del dato disponible.

In [26]:
limpio["race"] = limpio["race"].fillna("No informada")

pd.DataFrame({
    "pacientes": limpio["race"].value_counts(),
    "%": (100 * limpio["race"].value_counts(normalize=True)).round(2),
})

,pacientes,%
race,,
Caucasian,74220,74.71
AfricanAmerican,18772,18.90
No informada,2232,2.25
Hispanic,2017,2.03
Other,1471,1.48
Asian,628,0.63


## 2.7. Roles de las variables

Se declara explícitamente el papel de cada variable. Esta declaración se exporta y la usan los capítulos siguientes y el pipeline de modelado, lo que evita que cada notebook reconstruya sus propias listas y se desincronicen.

In [27]:
limpio = limpio.reset_index(drop=True)
registrar("5. Características derivadas y categorías agrupadas", limpio)

OBJETIVO = "readmitted_30"
IDENTIFICADOR = "patient_nbr"

NUMERICAS = [
    "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_outpatient", "number_emergency",
    "number_inpatient", "number_diagnoses", "n_farmacos_activos",
]
ORDINALES = ["age_ord"]
BINARIAS = ["hospitalizacion_previa"]
CATEGORICAS = sorted(
    c for c in limpio.columns
    if c not in NUMERICAS + ORDINALES + BINARIAS + [OBJETIVO, IDENTIFICADOR]
)

roles = {
    "objetivo": OBJETIVO,
    "identificador": IDENTIFICADOR,
    "numericas": NUMERICAS,
    "ordinales": ORDINALES,
    "binarias": BINARIAS,
    "categoricas": CATEGORICAS,
    "farmacos": FARMACOS,
    "etiquetas_edad": {str(k): v for k, v in ETIQUETAS_EDAD.items()},
    "random_state": RANDOM_STATE,
}

with open(DATOS_PROCESADOS / "roles_variables.json", "w", encoding="utf-8") as f:
    json.dump(roles, f, ensure_ascii=False, indent=2)

print(f"Numéricas   : {len(NUMERICAS)}")
print(f"Ordinales   : {len(ORDINALES)}")
print(f"Binarias    : {len(BINARIAS)}")
print(f"Categóricas : {len(CATEGORICAS)}")
print(f"Total de predictores: "
      f"{len(NUMERICAS) + len(ORDINALES) + len(BINARIAS) + len(CATEGORICAS)}")

Numéricas   : 9
Ordinales   : 1
Binarias    : 1
Categóricas : 27
Total de predictores: 38


In [28]:
inventario = pd.DataFrame({
    "tipo": limpio.dtypes.astype(str),
    "categorías": limpio.nunique(),
    "% faltantes": (100 * limpio.isna().mean()).round(2),
    "rol": [
        "objetivo" if c == OBJETIVO
        else "identificador" if c == IDENTIFICADOR
        else "numérica" if c in NUMERICAS
        else "ordinal" if c in ORDINALES
        else "binaria" if c in BINARIAS
        else "categórica"
        for c in limpio.columns
    ],
})
tabla(inventario, filas=45)

No queda ningún valor faltante en el conjunto final: los dos casos con ausencias sustantivas (`race` y la especialidad médica) se resolvieron con categorías explícitas, y las columnas con faltantes masivos se eliminaron. Esto simplifica el pipeline, que necesitará imputación únicamente como salvaguarda.

## 2.8. Partición de entrenamiento y prueba

La partición se crea **aquí**, antes de cualquier análisis bivariado, por una razón metodológica: explorar la relación entre predictores y desenlace sobre el conjunto completo y luego decidir qué variables o transformaciones usar introduce un sesgo de selección difícil de cuantificar. Los capítulos 3 a 5 trabajan solo con entrenamiento.

Se usa una partición estratificada 80/20 con semilla fija. La estratificación es indispensable dado el desbalance: con un 9 % de casos positivos, una partición aleatoria simple podría producir conjuntos con prevalencias apreciablemente distintas.

In [29]:
from sklearn.model_selection import StratifiedGroupKFold

# StratifiedGroupKFold con 5 particiones produce un corte 80/20 que respeta
# simultáneamente la estratificación por clase y la integridad de los grupos:
# todos los encuentros de un paciente quedan en el mismo lado de la partición.
particionador = StratifiedGroupKFold(n_splits=5, shuffle=True,
                                     random_state=RANDOM_STATE)
indices_train, indices_prueba = next(
    particionador.split(limpio, limpio[OBJETIVO], groups=limpio[IDENTIFICADOR])
)
entrenamiento = limpio.iloc[indices_train].reset_index(drop=True)
prueba = limpio.iloc[indices_prueba].reset_index(drop=True)

resumen_particion = pd.DataFrame({
    "registros": [len(entrenamiento), len(prueba), len(limpio)],
    "pacientes": [
        entrenamiento[IDENTIFICADOR].nunique(),
        prueba[IDENTIFICADOR].nunique(),
        limpio[IDENTIFICADOR].nunique(),
    ],
    "tasa clase positiva (%)": [
        100 * entrenamiento[OBJETIVO].mean(),
        100 * prueba[OBJETIVO].mean(),
        100 * limpio[OBJETIVO].mean(),
    ],
}, index=["Entrenamiento", "Prueba", "Total"]).round(3)

# Verificación: ningún paciente puede aparecer en ambos conjuntos.
solapamiento = set(entrenamiento[IDENTIFICADOR]) & set(prueba[IDENTIFICADOR])
assert not solapamiento, f"{len(solapamiento)} pacientes en ambas particiones"
print("Sin pacientes compartidos entre particiones")
resumen_particion

Sin pacientes compartidos entre particiones


,registros,pacientes,tasa clase positiva (%)
Entrenamiento,79473,56036,11.389
Prueba,19867,13951,11.391
Total,99340,69987,11.389


La verificación de solapamiento ya no es una formalidad: con todos los encuentros conservados, un paciente tiene en promedio 1.4 episodios y hasta 40 en el caso extremo, de modo que una partición por filas los repartiría entre entrenamiento y prueba. El `assert` detiene la ejecución si eso ocurre, y es la garantía de que la evaluación final se hace sobre pacientes que el modelo no ha visto nunca.

## 2.9. Exportación

In [31]:
# Verificaciones de integridad antes de exportar. Cualquier fallo detiene el
# notebook, de modo que un error no puede propagarse a los capítulos siguientes.
assert limpio.isna().sum().sum() == 0, "Quedan valores faltantes sin tratar"
assert set(limpio[OBJETIVO].unique()) == {0, 1}, "Target mal codificado"
assert "None" not in limpio[columnas_texto(limpio)].to_numpy(), \
    "Quedan cadenas 'None' que una lectura posterior convertiría en nulo"
assert not limpio["discharge_disposition_id"].isin(
    [mapeos["discharge_disposition_id"][c] for c in DESTINOS_EXCLUIDOS]
).any(), "Quedan altas incompatibles con una readmisión"

print(f"Verificaciones superadas: {len(limpio):,} encuentros de "
      f"{limpio[IDENTIFICADOR].nunique():,} pacientes")

Verificaciones superadas: 99,340 encuentros de 69,987 pacientes


In [32]:
destinos = [
    escribir_tabla(limpio, "diabetes_limpio"),
    escribir_tabla(entrenamiento, "diabetes_train"),
    escribir_tabla(prueba, "diabetes_test"),
]
guardar_resultado(pd.DataFrame(traza).set_index("etapa"), "trazabilidad_etl")

for destino in destinos:
    print(f"Exportado: {destino.name}")

pd.DataFrame(traza)

Exportado: diabetes_limpio.parquet
Exportado: diabetes_train.parquet
Exportado: diabetes_test.parquet


,etapa,filas,columnas,pacientes
0,0. Dataset crudo,101766,50,71518
1,1. Sin fallecimiento ni hospicio,99343,50,69990
2,2. Sin género inválido,99340,50,69987
3,4. Columnas descartadas y target derivado,99340,38,69987
4,5. Características derivadas y categorías agru...,99340,40,69987


### 2.10. Síntesis del capítulo

| Etapa | Resultado |
|---|---|
| Filtros aplicados | Fallecimiento y hospicio (2,423 filas), género inválido (3 filas), un encuentro por paciente (29,353 filas) |
| Columnas eliminadas | `weight`, `payer_code`, `encounter_id`, `readmitted`, nueve fármacos constantes |
| Variables derivadas | Tres grupos ICD-9, `age_ord`, `hospitalizacion_previa`, `n_farmacos_activos` |
| Categorías agrupadas | Umbral del 1 % aplicado a todas las categóricas: destino al alta de 21 a 9, especialidad de 72 a 11, tipo de admisión de 8 a 6, fuente de admisión de 17 a 7, más los niveles raros del panel farmacológico |
| Faltantes | Resueltos con categorías explícitas; `race` no se imputa por ser un atributo protegido |
| Conjunto final | 69,987 pacientes, 40 columnas (38 predictores, el target y el identificador), 8.98 % de clase positiva, sin valores faltantes |
| Partición | 55,989 de entrenamiento y 13,998 de prueba, estratificada, semilla 42 |
| Descartado | `visitas_previas`, por ser combinación lineal exacta de sus componentes |